In [ ]:

dbutils.widgets.text("catalog", "", "Catalog Name")
dbutils.widgets.text("schema", "", "Schema Name")
dbutils.widgets.text("warehouse", "", "Warehouse")
dbutils.widgets.text("govern_table", "", "AI Governance output table")

In [ ]:
from uuid import uuid4

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
warehouse = dbutils.widgets.get("warehouse")
govern_table = dbutils.widgets.get("govern_table")

run_id = f"{uuid4()}"

In [ ]:
from databricks.sdk.service.dashboards import GenieSpace
from governer.utils import task_genie

def govern_table_for_pii(space: GenieSpace, table: str):
    table_pii_govern_prompt=f"""
Execute following plan:

- Take a look at {table} table and find out columns which potentially can contain PII data (Personally identifiable information), but not marked with tag named "pii".  To get information about tags, inspect governer.infromation_schema.column_tags table.
- For each column you found on step 1, generate SQL statement to add tag "SET TAG ON COLUMN {table}.<column_name> `pii`=`<suspected_pii_type>`", where suspected_pii_type - type of PII data you suspect.
- For each column you found on step 1, express your confidence and provide number from 0.0 to 1.0, where 0.0 - you absolutely not confident and 1.0 - you 100% sure. Do not consider columns on which you not confident more than 0.1 
- Give me response as table with columns column_name, confidence and sql_statement, suspected_pii_type
    """
    return task_genie(space=space, task=table_pii_govern_prompt)

In [ ]:
from governer.utils import get_valid_govern_tables

spark.sql(f"use catalog {catalog}")
spark.sql(f"use schema {schema}")

tables = get_valid_govern_tables(catalog=catalog, schema=schema, tables=spark.catalog.listTables())



In [ ]:
from typing import List
from governer.utils import AiResponse, AiResponseWithEval

def parse_table_pii_results(input: AiResponse, table:str) -> List[AiResponseWithEval]:
    bad = [AiResponseWithEval(input.prompt, None, False, input.error, [], 0)]
    if not input.success:
        bad[0].text = input.text 
        return bad
    try:
        res = []
        if len(input.queries) != 1:
            return bad
        resultdf = spark.sql(input.queries[0])
        for result_row in resultdf.collect():
            txt = f"SET TAG ON COLUMN {table}.{result_row['column_name']} `pii`=`{result_row['suspected_pii_type']}`"
            res.append(AiResponseWithEval(prompt=input.prompt, text=txt, success=True, error=None, queries=[], eval=float(f"{result_row['confidence']}")))
        return res
    except Exception as e:
        bad[0].error = repr(e)
        return bad
    

In [ ]:
from governer.utils import trash_genie_workspace, get_genie_workspace, GovernStatus
from governer.schemas.governance.ai_govern_table import ai_govern_table
import datetime

genie_space_title="Genie Governer Space PII"

dsources = [f"{catalog}.information_schema.column_tags"]

for tbl in tables:
    dsources.append(tbl)

genie_space = get_genie_workspace(space_title=genie_space_title, warehouse_id=warehouse, tables=dsources)

res = []

for table in tables:
    response = govern_table_for_pii(space=genie_space, table=table)
    parsed = parse_table_pii_results(input=response, table=table)
    for resp in parsed:
        res.append({
            "record_id": f"{uuid4()}",
            "run_id": run_id, 
            "table_name": table, 
            "govern_success": resp.success, 
            "govern_error": resp.error,
            "evaluation": resp.eval,
            "sql": resp.text, 
            "status": GovernStatus.GENERATED.name, 
            "timestamp": datetime.datetime.now()
        })
        


trash_genie_workspace(genie_space)
govern = spark.createDataFrame(data=res,schema=ai_govern_table)
govern.write.mode("append").saveAsTable(govern_table)
